# Multi-Token Prediction Loss — 面试版

**难度：** Medium

**面试要点：** 纯手写交叉熵，不用 gather，用高级索引

In [ ]:
import torch

In [ ]:
# ✅ INTERVIEW

def multi_token_prediction_loss(hidden_states, heads, targets):
    B, S, D = hidden_states.shape
    N = len(heads)
    total_loss = 0.0

    for i, head in enumerate(heads):
        logits = hidden_states @ head  # (B, S, V)
        V = logits.shape[-1]

        # ---- 手写 log-softmax ----
        max_val = logits.max(dim=-1, keepdim=True).values
        shifted = logits - max_val
        log_sum_exp = torch.log(torch.exp(shifted).sum(dim=-1, keepdim=True))
        log_probs = shifted - log_sum_exp  # (B, S, V)

        # ---- 用高级索引代替 gather ----
        # 面试版: 不用 gather，用 torch.arange 构造索引
        tgt = targets[:, :, i]  # (B, S)
        # arange 构造 batch 和 seq 维度的索引
        b_idx = torch.arange(B).unsqueeze(1).expand(B, S)  # (B, S)
        s_idx = torch.arange(S).unsqueeze(0).expand(B, S)  # (B, S)
        # 高级索引: log_probs[b_idx, s_idx, tgt] 取出每个位置目标 token 的 log 概率
        log_p = log_probs[b_idx, s_idx, tgt]  # (B, S)

        total_loss = total_loss + (-log_p.mean())

    return total_loss / N

In [ ]:
torch.manual_seed(0)
B, S, D, V = 2, 5, 16, 10

hidden = torch.randn(B, S, D)
head_single = torch.randn(D, V)
targets_single = torch.randint(0, V, (B, S, 1))

mtp_loss = multi_token_prediction_loss(hidden, [head_single], targets_single)
print(f"MTP loss (N=1): {mtp_loss.item():.6f}")

# 验证: N=3
heads_3 = [torch.randn(D, V) for _ in range(3)]
targets_3 = torch.randint(0, V, (B, S, 3))
loss_3 = multi_token_prediction_loss(hidden, heads_3, targets_3)
print(f"MTP loss (N=3): {loss_3.item():.6f}")

In [ ]:
from torch_judge import check
check("multi_token_prediction")

## 📝 核心思路总结

1. **高级索引替代 gather**：`log_probs[batch_idx, seq_idx, target]` 直接取值
2. **手写 log-softmax**：减最大值 → exp → sum → log → 减
3. **broadcast 索引**：`arange` + `expand` 构造二维索引矩阵